#Source 1- Load the internal dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np

In [2]:
# Load the cleaned operational dataset
df_ops = pd.read_csv("cleaned_ops.csv")

# Convert timestamp to datetime
df_ops["timestamp"] = pd.to_datetime(df_ops["timestamp"])

# Preview the data
df_ops.head()

,timestamp,Zone,Shift,Pressure_PSI,Temperature_C,Flow_Rate_LPM
0,2026-06-25 00:10:00,ZONE_SOUTH,Morning,222.844248,53.393063,760.039387
1,2026-06-25 00:22:00,ZONE_EAST,Night,248.259118,53.228993,894.431039
2,2026-06-25 00:24:00,ZONE_CENTRAL,Afternoon,220.866819,57.467330,981.742087
3,2026-06-25 00:26:00,ZONE_SOUTH,Night,166.868033,57.311710,945.984000
4,2026-06-25 00:28:00,ZONE_WEST,Morning,179.766447,56.560253,1180.532611


In [3]:
df_ops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1839 entries, 0 to 1838
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      1839 non-null   datetime64[ns]
 1   Zone           1839 non-null   object        
 2   Shift          1820 non-null   object        
 3   Pressure_PSI   1839 non-null   float64       
 4   Temperature_C  1839 non-null   float64       
 5   Flow_Rate_LPM  1839 non-null   float64       
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 86.3+ KB


Source 2: External API.

In [4]:
import requests

In [5]:
# Fetch current weather data for Nairobi
url = "https://wttr.in/Nairobi?format=j1"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    weather = response.json()

    current_weather = {
        "date": pd.Timestamp.today().normalize(),
        "temperature_C": float(weather["current_condition"][0]["temp_C"]),
        "humidity": int(weather["current_condition"][0]["humidity"]),
        "precipitation_mm": float(weather["current_condition"][0]["precipMM"])
    }

    df_weather = pd.DataFrame([current_weather])

    print("Weather data fetched successfully!")
    display(df_weather)

except requests.exceptions.RequestException as e:
    print(f"API Error: {e}")

Weather data fetched successfully!


,date,temperature_C,humidity,precipitation_mm
0,2026-07-15,15.0,94,0.0


#Source 3: SQLite Database.

In [6]:
from sqlalchemy import create_engine

# Create SQLite database
engine = create_engine("sqlite:///operations.db")

# Create a holiday calendar
holiday_data = pd.DataFrame({
    "date": pd.to_datetime([
        "2026-06-25",
        "2026-06-26",
        "2026-06-27",
        "2026-06-28",
        "2026-06-29"
    ]),
    "holiday_type": [
        "Working Day",
        "Working Day",
        "Weekend",
        "Weekend",
        "Working Day"
    ]
})

# Save to SQLite
holiday_data.to_sql("Holiday_Calendar", engine, if_exists="replace", index=False)

print("Database created successfully!")

Database created successfully!


#Date column to operational data

In [7]:
# Extract date from timestamp
df_ops["date"] = df_ops["timestamp"].dt.date

# Save operations data to SQLite
df_ops.to_sql("Operations", engine, if_exists="replace", index=False)

1839

#SQL Query (JOIN + GROUP BY)

In [8]:
query = """
SELECT
    h.holiday_type,
    COUNT(o.timestamp) AS total_records,
    ROUND(AVG(o.Flow_Rate_LPM), 2) AS avg_flow_rate
FROM Operations o
LEFT JOIN Holiday_Calendar h
ON DATE(o.date) = DATE(h.date)
GROUP BY h.holiday_type;
"""

df_sql = pd.read_sql(query, engine)

df_sql

,holiday_type,total_records,avg_flow_rate
0,None,536,1000.67
1,Weekend,530,989.01
2,Working Day,773,989.84
